<a href="https://colab.research.google.com/github/Ayomide-baga/CMP7005-PRAC1/blob/main/CMP7005_PRAC1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CMP7005 Programming for Data Analysis — PRAC1
## From Data to Application Development
### Beijing Air Quality Analysis

**Student ID:** ST20349610  
**Module Leader:** aprasad@cardiffmet.ac.uk

## Research Context

This analysis investigates the spatial and temporal patterns of air pollution across Beijing's urban-suburban gradient, with a focus on PM2.5 as the primary health-relevant pollutant. The key questions guiding this work are:

1. How do pollution levels differ between urban and suburban monitoring stations?
2. What meteorological and temporal factors drive PM2.5 variability?
3. Can PM2.5 concentrations be predicted from other pollutant and weather measurements?

These questions inform every stage of the analysis — from station selection (Task 1), through exploratory analysis (Task 2), to predictive modelling (Task 3) and the interactive application (Task 4).

## Task 1: Data Selection & Handling

### 1.1 Station Selection

For this analysis, **two urban (inner)** and **two suburban (outer)** monitoring stations were selected from Beijing's 12 nationally controlled air quality monitoring sites.

**Urban stations selected: Dongsi and Tiantan**  
**Suburban stations selected: Dingling and Huairou**

Xu and Zhang (2020) classify eight of the twelve Beijing monitoring stations as urban (Aotizhongxin, Dongsi, Guanyuan, Nongzhanguan, Tiantan, Wanshouxigong, Gucheng, and Haidian) and four as suburban (Changping, Dingling, Huairou, and Shunyi).

**Dongsi** is located in Beijing's Dongcheng District within the capital core zone (Z4), surrounded by dense residential neighbourhoods and heavy traffic, making it highly representative of inner-city pollution (Batterman et al., 2016).

**Tiantan** (Temple of Heaven) sits in the southern urban area, capturing pollution from local traffic and regional transport from the industrialised south (Guo et al., 2019).

**Dingling** is classified as a background/contrast station in a mountainous, forested area far from urban emissions (Batterman et al., 2016).

**Huairou** lies within the ecological conservation zone, representing the cleanest conditions among the 12 stations (Batterman et al., 2016).

This selection enables comparison across the urban–suburban gradient.

### 1.2 Setting Up the Environment

In [1]:
# Importing the necessary libraries for data analysis
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

In [2]:
! git config --global user.name "Ayomide-baga"
! git config --global user.email "bamigbopaayomide@gmail.com"

In [3]:
username = "Ayomide-baga"
repo = "CMP7005-PRAC1"

In [4]:
! git clone https://@github.com/{username}/{repo}

Cloning into 'CMP7005-PRAC1'...
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 9 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (9/9), 2.77 MiB | 4.72 MiB/s, done.


In [5]:
%cd {repo}

/content/CMP7005-PRAC1


In [6]:
%ls

PRSA_Data_Dingling_20130301-20170228.csv
PRSA_Data_Dongsi_20130301-20170228.csv
PRSA_Data_Huairou_20130301-20170228.csv
PRSA_Data_Tiantan_20130301-20170228.csv
README.md


### 1.3 Loading and Merging the Datasets

In [7]:
# Load all station CSV files from the repository
city_files = glob.glob("*.csv")

dataframes = []
for file_name in city_files:
    city_df = pd.read_csv(file_name)
    dataframes.append(city_df)
    print(f"Loaded: {file_name}")

# Merge into a single unified dataset
df = pd.concat(dataframes, ignore_index=True)

# Confirmation
print(f"\nSUCCESS: Combined {len(city_files)} city files into one file with {len(df)} total rows")
print(f"Columns: {df.shape[1]}")
print(f"Stations: {df['station'].unique()}")

Loaded: PRSA_Data_Dongsi_20130301-20170228.csv
Loaded: PRSA_Data_Tiantan_20130301-20170228.csv
Loaded: PRSA_Data_Dingling_20130301-20170228.csv
Loaded: PRSA_Data_Huairou_20130301-20170228.csv

SUCCESS: Combined 4 city files into one file with 140256 total rows
Columns: 18
Stations: ['Dongsi' 'Tiantan' 'Dingling' 'Huairou']


In [8]:
# Create a proper datetime column from individual time components
df['datetime'] = pd.to_datetime(df[['year', 'month', 'day', 'hour']])

# Set as index for time-series analysis
df.set_index('datetime', inplace=True)

# Verify the date range and structure
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"\nRecords per station:")
print(df['station'].value_counts())
df.head()

Date range: 2013-03-01 00:00:00 to 2017-02-28 23:00:00

Records per station:
station
Dongsi      35064
Tiantan     35064
Dingling    35064
Huairou     35064
Name: count, dtype: int64


,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
datetime,,,,,,,,,,,,,,,,,,
2013-03-01 00:00:00,1,2013,3,1,0,9.0,9.0,3.0,17.0,300.0,89.0,-0.5,1024.5,-21.4,0.0,NNW,5.7,Dongsi
2013-03-01 01:00:00,2,2013,3,1,1,4.0,4.0,3.0,16.0,300.0,88.0,-0.7,1025.1,-22.1,0.0,NW,3.9,Dongsi
2013-03-01 02:00:00,3,2013,3,1,2,7.0,7.0,NaN,17.0,300.0,60.0,-1.2,1025.3,-24.6,0.0,NNW,5.3,Dongsi
2013-03-01 03:00:00,4,2013,3,1,3,3.0,3.0,5.0,18.0,NaN,NaN,-1.4,1026.2,-25.5,0.0,N,4.9,Dongsi
2013-03-01 04:00:00,5,2013,3,1,4,3.0,3.0,7.0,NaN,200.0,84.0,-1.9,1027.1,-24.5,0.0,NNW,3.2,Dongsi


In [9]:
# Save merged dataset with datetime index
df.to_csv('merged_air_quality.csv')
print("Merged dataset saved as 'merged_air_quality.csv'")

Merged dataset saved as 'merged_air_quality.csv'
